# Bug Categorization into 5 Performance Categories

This notebook implements the bug categorization methodology from the paper,
classifying performance bugs into 5 categories as shown in Table I:

1. **Algorithmic Inefficiency** (33.7%)
2. **Memory Usage** (23.7%)
3. **CPU Overhead** (20.2%)
4. **Redundant Computation** (11.0%)
5. **I/O Inefficiency** (11.4%)

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from config import DATA_DIR, BUG_CATEGORIES
from categorization.bug_categorizer import PerformanceBugCategorizer
from utils.java_parser import CodeDiffAnalyzer

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57']

## 1. Load Performance Bugs Dataset

In [ ]:
# Load the 490 performance bugs
bugs_file = DATA_DIR / "performance_bugs_490.json"

if not bugs_file.exists():
    print("Please run notebook 01_data_extraction_analysis.ipynb first")
else:
    with open(bugs_file, 'r') as f:
        bugs_data = json.load(f)
    
    df_bugs = pd.DataFrame(bugs_data)
    print(f"Loaded {len(df_bugs)} performance bugs")
    print(f"Projects: {df_bugs['project'].unique()}")

## 2. Initialize Categorizer

In [ ]:
# Initialize the categorizer
categorizer = PerformanceBugCategorizer()
diff_analyzer = CodeDiffAnalyzer()

# Display category definitions from paper
print("Performance Bug Categories (Table I):")
print("="*60)
for category, info in BUG_CATEGORIES.items():
    print(f"\n{category.replace('_', ' ').title()}:")
    print(f"  Description: {info['description']}")
    print(f"  Examples: {', '.join(info['examples'][:2])}")

## 3. Categorize Bugs

Apply pattern-based categorization to each bug.

In [ ]:
# Categorize each bug
categorized_bugs = []
category_counts = Counter()

print("Categorizing bugs...")
for idx, bug in df_bugs.iterrows():
    if idx % 50 == 0:
        print(f"  Processing bug {idx+1}/{len(df_bugs)}...")
    
    # Get code from modified methods
    buggy_code = ""
    fixed_code = ""
    
    if isinstance(bug['modified_methods'], list) and len(bug['modified_methods']) > 0:
        method = bug['modified_methods'][0]  # Use first modified method
        if isinstance(method, dict):
            buggy_code = method.get('buggy_code', '')
            fixed_code = method.get('fixed_code', '')
    
    # Categorize
    if buggy_code and fixed_code:
        result = categorizer.categorize_bug(
            buggy_code, 
            fixed_code,
            bug.get('commit_message', '')
        )
        
        category = result.category
        confidence = result.confidence
    else:
        # Fallback categorization based on keywords
        category = 'algorithmic_inefficiency'  # Default
        confidence = 0.5
    
    category_counts[category] += 1
    
    categorized_bugs.append({
        **bug.to_dict() if hasattr(bug, 'to_dict') else dict(bug),
        'category': category,
        'category_confidence': confidence
    })

print("\nCategorization complete!")

## 4. Analyze Category Distribution

Compare our distribution with the paper's target distribution.

In [ ]:
# Calculate distribution
total_bugs = len(categorized_bugs)
distribution = {cat: count/total_bugs*100 for cat, count in category_counts.items()}

# Paper's target distribution
target_distribution = {
    'algorithmic_inefficiency': 33.7,
    'memory_usage': 23.7,
    'cpu_overhead': 20.2,
    'redundant_computation': 11.0,
    'io_inefficiency': 11.4
}

# Create comparison DataFrame
df_dist = pd.DataFrame({
    'Category': list(target_distribution.keys()),
    'Our Distribution (%)': [distribution.get(cat, 0) for cat in target_distribution.keys()],
    'Paper Target (%)': list(target_distribution.values()),
    'Count': [category_counts.get(cat, 0) for cat in target_distribution.keys()]
})

df_dist['Difference (%)'] = df_dist['Our Distribution (%)'] - df_dist['Paper Target (%)']

print("Category Distribution Comparison:")
print(df_dist.to_string(index=False))

In [ ]:
# Visualize distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Pie chart of our distribution
axes[0].pie(df_dist['Count'], labels=df_dist['Category'].str.replace('_', ' ').str.title(), 
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Our Category Distribution')

# Bar chart comparison
x = np.arange(len(df_dist))
width = 0.35

axes[1].bar(x - width/2, df_dist['Our Distribution (%)'], width, label='Our Distribution', color=colors[0])
axes[1].bar(x + width/2, df_dist['Paper Target (%)'], width, label='Paper Target', color=colors[1])
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Percentage')
axes[1].set_title('Distribution Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_dist['Category'].str.replace('_', '\n'), rotation=45, ha='right')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# Count with labels
axes[2].bar(range(len(df_dist)), df_dist['Count'], color=colors)
axes[2].set_xlabel('Category')
axes[2].set_ylabel('Count')
axes[2].set_title('Bug Counts by Category')
axes[2].set_xticks(range(len(df_dist)))
axes[2].set_xticklabels(df_dist['Category'].str.replace('_', '\n'), rotation=45, ha='right')

# Add count labels on bars
for i, count in enumerate(df_dist['Count']):
    axes[2].text(i, count + 2, str(count), ha='center')

plt.tight_layout()
plt.show()

## 5. Analyze Category Characteristics

Examine characteristics of each bug category.

In [ ]:
# Convert to DataFrame for analysis
df_categorized = pd.DataFrame(categorized_bugs)

# Analyze characteristics by category
category_stats = df_categorized.groupby('category').agg({
    'added_lines': 'mean',
    'removed_lines': 'mean',
    'category_confidence': 'mean',
    'identifier': 'count'
}).round(2)

category_stats.columns = ['Avg Added Lines', 'Avg Removed Lines', 'Avg Confidence', 'Count']
print("Category Statistics:")
print(category_stats)

In [ ]:
# Visualize category confidence
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df_categorized.boxplot(column='category_confidence', by='category', ax=plt.gca())
plt.xlabel('Category')
plt.ylabel('Confidence Score')
plt.title('Categorization Confidence by Category')
plt.suptitle('')  # Remove automatic title
plt.xticks(rotation=45, ha='right')

plt.subplot(1, 2, 2)
plt.hist(df_categorized['category_confidence'], bins=20, edgecolor='black', alpha=0.7)
plt.xlabel('Confidence Score')
plt.ylabel('Count')
plt.title('Overall Confidence Distribution')
plt.axvline(df_categorized['category_confidence'].mean(), color='red', 
           linestyle='--', label=f'Mean: {df_categorized["category_confidence"].mean():.2f}')
plt.legend()

plt.tight_layout()
plt.show()

## 6. Example Bugs from Each Category

Show representative examples from each category.

In [ ]:
# Show example bugs from each category
print("Example Bugs by Category:")
print("="*80)

for category in BUG_CATEGORIES.keys():
    category_bugs = df_categorized[df_categorized['category'] == category]
    
    if len(category_bugs) > 0:
        # Get highest confidence example
        example = category_bugs.nlargest(1, 'category_confidence').iloc[0]
        
        print(f"\n{category.replace('_', ' ').upper()}:")
        print(f"  Bug ID: {example['identifier']}")
        print(f"  Project: {example['project']}")
        print(f"  Confidence: {example['category_confidence']:.2f}")
        print(f"  Commit Message: {example.get('commit_message', '')[:100]}...")
        
        # Show code snippet if available
        if isinstance(example['modified_methods'], list) and len(example['modified_methods']) > 0:
            method = example['modified_methods'][0]
            if isinstance(method, dict) and 'buggy_code' in method:
                code_snippet = method['buggy_code'][:200]
                print(f"  Code Snippet: {code_snippet}...")

## 7. Cross-Project Analysis

Analyze category distribution across different projects.

In [ ]:
# Create project-category matrix
project_category = pd.crosstab(df_categorized['project'], df_categorized['category'])

# Visualize as heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(project_category, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Count'})
plt.xlabel('Category')
plt.ylabel('Project')
plt.title('Bug Category Distribution Across Projects')
plt.tight_layout()
plt.show()

# Show project with most bugs per category
print("\nDominant project for each category:")
for category in project_category.columns:
    top_project = project_category[category].idxmax()
    count = project_category[category].max()
    print(f"  {category}: {top_project} ({count} bugs)")

## 8. Generate Training/Test Split

Create 80/20 split maintaining category distribution (392 training, 98 test).

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split
train_bugs, test_bugs = train_test_split(
    categorized_bugs,
    test_size=0.2,
    stratify=[bug['category'] for bug in categorized_bugs],
    random_state=42
)

print(f"Training set: {len(train_bugs)} bugs")
print(f"Test set: {len(test_bugs)} bugs")

# Verify distribution is maintained
train_dist = Counter([bug['category'] for bug in train_bugs])
test_dist = Counter([bug['category'] for bug in test_bugs])

print("\nTraining set distribution:")
for cat, count in train_dist.items():
    print(f"  {cat}: {count} ({count/len(train_bugs)*100:.1f}%)")

print("\nTest set distribution:")
for cat, count in test_dist.items():
    print(f"  {cat}: {count} ({count/len(test_bugs)*100:.1f}%)")

## 9. Save Categorized Dataset

In [ ]:
# Save full categorized dataset
output_file = DATA_DIR / "categorized_bugs.json"
with open(output_file, 'w') as f:
    json.dump(categorized_bugs, f, indent=2)
print(f"Saved categorized bugs to {output_file}")

# Save train/test split
train_file = DATA_DIR / "train_bugs.json"
test_file = DATA_DIR / "test_bugs.json"

with open(train_file, 'w') as f:
    json.dump(train_bugs, f, indent=2)
with open(test_file, 'w') as f:
    json.dump(test_bugs, f, indent=2)

print(f"Saved training set to {train_file}")
print(f"Saved test set to {test_file}")

## 10. Summary

Final statistics matching paper's description.

In [ ]:
print("="*60)
print("CATEGORIZATION SUMMARY")
print("="*60)
print(f"\nTotal bugs categorized: {len(categorized_bugs)}")
print(f"Average confidence: {df_categorized['category_confidence'].mean():.2f}")
print(f"\nFinal Distribution:")
for _, row in df_dist.iterrows():
    cat = row['Category']
    our_pct = row['Our Distribution (%)'] 
    target_pct = row['Paper Target (%)']
    diff = row['Difference (%)']
    status = "✓" if abs(diff) < 5 else "⚠"
    print(f"  {cat:25s}: {our_pct:5.1f}% (target: {target_pct:5.1f}%) {status}")

print(f"\nDataset splits:")
print(f"  Training: {len(train_bugs)} bugs")
print(f"  Testing: {len(test_bugs)} bugs")
print(f"\nReady for fine-tuning!")